In [38]:
from langchain_community.document_loaders import FileSystemBlobLoader
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import PyMuPDFParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from getpass import getpass
import os

In [39]:
DATA_DIR = Path("data")

In [19]:
loader = GenericLoader(
    blob_loader=FileSystemBlobLoader(
        path=DATA_DIR,
        glob="*.pdf",
    ),
    blob_parser=PyMuPDFParser(mode='page')
)

docs = loader.load() # TODO : pour le moment, un doc = une page et pas de chunking supplementaire

In [40]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

embeddings = OpenAIEmbeddings(
    model="Qwen/Qwen3-Embedding-8B",
    base_url="https://inference.rcp.epfl.ch/v1"
)

In [23]:
qdrant = QdrantVectorStore.from_documents(
    docs,
    embeddings,
    url="http://localhost:6333",
    collection_name="basic_pdf_split_pages",
)

In [34]:
query = "Comment les responsabilités en matière de contrôle interne sont-elles réparties entre la direction, les unités opérationnelles et les fonctions de supervision (audit, contrôle des risques) au sein de l'EPFL?"
found_docs = qdrant.similarity_search_with_score(query)
found_docs

[(Document(metadata={'producer': 'Microsoft® Word pour Microsoft\xa0365', 'creator': 'Microsoft® Word pour Microsoft\xa0365', 'creationdate': '2025-03-05T09:20:53+01:00', 'source': 'data/fr_8tMCzmETABTBbFD3H_LEX_1.10.1.pdf', 'file_path': 'data/fr_8tMCzmETABTBbFD3H_LEX_1.10.1.pdf', 'total_pages': 11, 'format': 'PDF 1.7', 'title': '', 'author': 'Microsoft Office User', 'subject': '', 'keywords': '', 'moddate': '2025-03-05T09:20:53+01:00', 'trapped': '', 'modDate': "D:20250305092053+01'00'", 'creationDate': "D:20250305092053+01'00'", 'page': 9, '_id': '550cbed5-6f92-4961-85a5-993a0c457353', '_collection_name': 'basic_pdf_split_pages'}, page_content='Version 1.5 \n10/11 \nLEX 1.10.1 du 15 juillet 2014,  \n \nDirective concernant le sponsoring \nétat au 1er janvier 2025 \n \net le mécénat à l’EPFL'),
  0.7711076),
 (Document(metadata={'producer': 'Microsoft® Word pour Microsoft\xa0365', 'creator': 'Microsoft® Word pour Microsoft\xa0365', 'creationdate': '2025-03-12T09:35:55+01:00', 'source'

In [26]:
# TODO : tester avec d'autres parametres de chunking ou voir https://docs.langchain.com/oss/python/integrations/splitters

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=20,
    separators=["\n \n", "\n\n", "\n", ". ", " ", ""],
    add_start_index=True
)

chunks = text_splitter.split_documents(docs)